# Demo preparation: mẫu phân tích và chuỗi tháng

Đầu vào là hai clean. Tạo lại năm bảng dẫn xuất bằng chính các builder của project; không làm sạch raw lần nữa.

Chạy từ trên xuống bằng **Restart Kernel → Run All**, hoặc Shift+Enter từng cell. Thuật toán dùng lại từ `source/`; không sửa các CSV nghiên cứu đã có. Xem `notebooks/README.md` để cấu hình đường dẫn/kernel.

## Đường dẫn và môi trường
Kiểm tra các đường dẫn trước khi chạy. Các notebook độc lập: mặc định đọc clean/preparation đã lưu; không yêu cầu chạy notebook trước để có biến trong RAM.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                     if (p/'source/preprocessing').is_dir() and (p/'notebooks').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the repository root or notebooks directory.')
for folder in [PROJECT_ROOT, PROJECT_ROOT/'notebooks', PROJECT_ROOT/'source/preprocessing']:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
from demo_support import (configure, show_paths, demo_directory, save_table, save_json,
                          show_source, show_png, environment, compare_values, read_series, series_plot)
PATHS = configure(PROJECT_ROOT)
# Nếu cần, sửa PATHS.population_raw / history_raw / population_clean / history_clean / preparation tại đây.
show_paths(PATHS)
print('Python kernel:', sys.executable)


,setting,path,exists
0,project,D:\Phân tích kinh doanh_2627\GitHub\IS403.R14....,True
1,data_root,D:\Phân tích kinh doanh_2627,True
2,population_raw,D:\Phân tích kinh doanh_2627\archive\Electric_...,True
3,history_raw,D:\Phân tích kinh doanh_2627\Electric_Vehicle_...,True
4,population_clean,D:\Phân tích kinh doanh_2627\archive\EV_Popula...,True
5,history_clean,D:\Phân tích kinh doanh_2627\Electric_Vehicle_...,True
6,preparation,D:\Phân tích kinh doanh_2627\Du_lieu_dan_xuat\...,True
7,output_root,D:\Phân tích kinh doanh_2627\Du_lieu_dan_xuat\...,True


Python kernel: C:\Users\ASUS\AppData\Local\Programs\Python\Python312\python.exe


In [2]:
OUT = demo_directory(PATHS, 'preparation')
print('Output demo:', OUT)
display(environment())


Output demo: D:\Phân tích kinh doanh_2627\Du_lieu_dan_xuat\notebook_demo\preparation_20261116_035855_582c5cf9


,package,version,status
0,pandas,3.0.5,Đã tìm thấy
1,numpy,2.5.3,Đã tìm thấy
2,scipy,1.18.1,Đã tìm thấy
3,matplotlib,3.11.2,Đã tìm thấy
4,statsmodels,0.15.0,Đã tìm thấy
5,scikit-learn,1.9.1,Đã tìm thấy
6,xgboost,3.4.1,Đã tìm thấy
7,ipykernel,7.3.0,Đã tìm thấy


## Đọc và kiểm tra clean
Các loader kiểm tra schema, WA, nhãn và đồng nhất số đếm.

In [3]:
from source.preparation.build_analysis_samples import load_population, build_analysis_samples
from source.preparation.build_monthly_series import load_history, build_monthly_series, write_outputs
population = load_population(PATHS.population_clean)
history = load_history(PATHS.history_clean)
display(pd.DataFrame({'dataset':['Population','History'],'rows':[len(population),len(history)]}))
show_source(build_monthly_series)


,dataset,rows
0,Population,134726
1,History,6474


def build_monthly_series(history: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Aggregate validated History counts, then calculate ratios and splits."""
    monthly = history.groupby('Date', sort=True)[list(COUNT_MAP)].sum().rename(columns=COUNT_MAP).reset_index()
    monthly = monthly.rename(columns={'Date': 'date'})
    monthly['ev_pct'] = 100 * monthly['ev_total'] / monthly['total_vehicles']
    monthly['split'] = monthly['date'].le(pd.Timestamp(TRAIN_END)).map({True: 'train', False: 'test'})
    require(monthly['ev_total'].gt(0).all(), 'Monthly EV totals must be positive.')
    require(monthly['split'].value_counts().to_dict() == {'train': 71, 'test': 12}, 'Expected 71 train months and 12 test months.')
    require(monthly.loc[monthly['split'].eq('test'), 'date'].min() == pd.Timestamp(TEST_START), 'Incorrect first test month.')
    monthly['date'] = monthly['date'].dt.strftime('%Y-%m-%d')
    summary = {
        'history_rows': int(len(history)), 'counties': int(history['County'].nunique()),
        'use_groups': sorted(history['Vehicle Primary Use'].unique().tolist()),
        'rows_per_month': 78, 'monthly_rows': int(len(monthly)),
        'first_month': FIRST_MONTH, 'last_month': LAST_MONTH,
        'train': {'start': FIRST_MONTH, 'end': TRAIN_END, 'months': 71},
        'test': {'start': TEST_START, 'end': LAST_MONTH, 'months': 12},
        'columns': monthly.columns.tolist(), 'count_unit': 'vehicles', 'ev_pct_unit': 'percent',
        'aggregation': 'Sum count columns across counties and both use groups; ev_pct = 100 * ev_total / total_vehicles.',
        'features_created': False,
    }
    return monthly, summary

## Tổng hợp chuỗi toàn bang
Tổng hợp số lượng theo tháng rồi tính lại EV/total; không lấy trung bình phần trăm quận.

In [4]:
monthly, monthly_summary = build_monthly_series(history)
display(monthly.head())
display(monthly.tail())
assert len(monthly)==83
compare_values('EV = BEV + PHEV', monthly.ev_total, monthly.bev+monthly.phev)
compare_values('EV percentage', monthly.ev_pct, 100*monthly.ev_total/monthly.total_vehicles)


,date,bev,phev,ev_total,non_ev,total_vehicles,ev_pct,split
0,2017-01-31,14359,7436,21795,5918988,5940783,0.366871,train
1,2017-02-28,14832,7638,22470,5908594,5931064,0.378853,train
2,2017-03-31,15223,7839,23062,5908100,5931162,0.388828,train
3,2017-04-30,15707,8033,23740,5903511,5927251,0.400523,train
4,2017-05-31,16102,8247,24349,5905661,5930010,0.410606,train


,date,bev,phev,ev_total,non_ev,total_vehicles,ev_pct,split
78,2023-07-31,110114,32423,142537,5985419,6127956,2.326012,test
79,2023-08-31,115442,33288,148730,5966752,6115482,2.432024,test
80,2023-09-30,118597,34093,152690,5935089,6087779,2.508140,test
81,2023-10-31,122372,34789,157161,5886324,6043485,2.600503,test
82,2023-11-30,124353,35125,159478,5794576,5954054,2.678478,test


,check,passed,max_absolute_difference
0,EV = BEV + PHEV,True,0.0


,check,passed,max_absolute_difference
0,EV percentage,True,0.0


## Quan sát train/test
71 tháng train đến 11/2022 và 12 tháng test từ 12/2022. Split không phải chọn cấu hình bằng test.

In [5]:
display(monthly.groupby('split',sort=False).agg(months=('date','size'),start=('date','min'),end=('date','max')))
display(monthly.iloc[68:74])
assert monthly['split'].value_counts().to_dict()=={'train':71,'test':12}


,months,start,end
split,,,
train,71,2017-01-31,2022-11-30
test,12,2022-12-31,2023-11-30


,date,bev,phev,ev_total,non_ev,total_vehicles,ev_pct,split
68,2022-09-30,79504,26380,105884,6068191,6174075,1.714978,train
69,2022-10-31,82725,26723,109448,6061648,6171096,1.773559,train
70,2022-11-30,85877,27071,112948,6057519,6170467,1.830461,train
71,2022-12-31,88165,27500,115665,6050175,6165840,1.875900,test
72,2023-01-31,91351,28026,119377,6044641,6164018,1.936675,test
73,2023-02-28,93751,28571,122322,6035609,6157931,1.986414,test


## Tạo mẫu H1–H4
Range, CAFV, cơ cấu quận và snapshot có điều kiện lọc/mẫu số khác nhau.

In [6]:
samples, sample_summary = build_analysis_samples(population, history)
display(sample_summary)
for name, frame in samples.items():
    print(name, frame.shape)
    display(frame.head(3))


,sample,source,rows,vehicle_count,counties,condition
0,h1_primary,Population clean,77857,77857,39,electric_range is not missing
1,h1_primary_bev,Population clean,46806,46806,39,valid Range; vehicle_type=BEV
2,h1_primary_phev,Population clean,31051,31051,39,valid Range; vehicle_type=PHEV
3,h1_sensitivity,Population clean,65779,65779,39,valid Range; Model Year <= 2020
4,h2_primary,Population clean,77857,77857,39,Range and Model Year are not missing
...,...,...,...,...,...,...
8,h3_sensitivity,Population clean,134726,134726,39,"all CAFV labels, including Unknown"
9,h4a_primary,Population clean,39,134726,39,all counties; rows are county aggregates
10,h4a_sensitivity,Population clean,36,134683,36,county ev_total >= 30
11,h4b_primary,History clean,39,5954054,39,date=2023-11-30; sum Passenger and Truck; test...


range_analysis_sample.csv (77857, 7)


,source_record,county,vehicle_type,model_year,electric_range,h1_year_le_2020,h2_valid
0,1,Adams,BEV,2020.0,266.0,True,True
1,2,Adams,PHEV,2020.0,32.0,True,True
2,4,Adams,BEV,2011.0,73.0,True,True


cafv_analysis_sample.csv (134726, 5)


,source_record,county,vehicle_type,cafv_status,cafv_known
0,1,Adams,BEV,Clean Alternative Fuel Vehicle Eligible,True
1,2,Adams,PHEV,Clean Alternative Fuel Vehicle Eligible,True
2,3,Adams,BEV,Eligibility unknown as battery range has not b...,False


county_vehicle_type_counts.csv (39, 5)


,county,bev,phev,ev_total,included_h4a_sensitivity
0,Adams,27,12,39,True
1,Asotin,35,20,55,True
2,Benton,1135,552,1687,True


county_ev_snapshots.csv (78, 8)


,date,county,bev,phev,ev_total,non_ev,total_vehicles,ev_pct
0,2017-01-31,Adams,1,1,2,17251,17253,0.011592
1,2017-01-31,Asotin,4,3,7,18805,18812,0.037210
2,2017-01-31,Benton,129,129,258,167665,167923,0.153642


## Đối chiếu và xuất bản demo
Các bảng tái tạo phải khớp preparation đã lưu. File dẫn xuất vẫn nằm ngoài repo.

In [7]:
for name, frame in {'statewide_monthly.csv':monthly, **samples}.items():
    reference = pd.read_csv(PATHS.preparation/name)
    pd.testing.assert_frame_equal(frame, reference, check_dtype=False)
outputs = {OUT/name:frame for name,frame in {'statewide_monthly.csv':monthly, **samples}.items()}
outputs.update({OUT/'sample_summary.csv':sample_summary, OUT/'monthly_series_summary.json':monthly_summary})
write_outputs(outputs, [PATHS.population_clean, PATHS.history_clean])
print('Đã đối chiếu và xuất 5 CSV dữ liệu cùng 2 tóm tắt:', OUT)


Đã đối chiếu và xuất 5 CSV dữ liệu cùng 2 tóm tắt: D:\Phân tích kinh doanh_2627\Du_lieu_dan_xuat\notebook_demo\preparation_20261116_035855_582c5cf9
